# 엔트로피 가중치 기반 교통 지표 점수 산출

이 노트북은 네 개의 교통 관련 최종 지표를 따로 사용해 엔트로피 가중치를 계산하고, 자치구별 교통 지표 점수를 25점 만점으로 산출합니다.

사용 지표는 다음과 같습니다.

- 대중교통점수: `transport_stress_score`
- 출퇴근스트레스점수: `commute_stress_score`
- 주차점수: `parking_score_100`
- 민원점수: `complaint_score_100`

모든 점수는 높을수록 교통 문제가 큰 방향으로 해석합니다.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'branch_SG' / 'traffic_analystic').exists():
            return path
    raise FileNotFoundError('프로젝트 루트(branch_SG/traffic_analystic 포함)를 찾을 수 없습니다.')

BASE_DIR = find_project_root()
TRAFFIC_DIR = BASE_DIR / 'branch_SG' / 'traffic_analystic'
DATA_DIR = TRAFFIC_DIR / 'traffic_analystic_data'
OUT_DIR = TRAFFIC_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BUS_PATH = DATA_DIR / 'bus_gu_wide_202605071519.csv'
SUBWAY_PATH = DATA_DIR / 'subway_gu_wide_202605071519.csv'
MONTHLY_PATH = DATA_DIR / 'monthly_scores_문헌기반_서울_202605071520.csv'
POPULATION_PATH = DATA_DIR / 'Garbage_collection_status.csv'
PARKING_COMPLAINT_PATH = OUT_DIR / 'gu_parking_complaint_score_summary.csv'

FINAL_SCORE_PATH = OUT_DIR / 'gu_traffic_entropy_score_25.csv'
WEIGHT_PATH = OUT_DIR / 'traffic_entropy_weights.csv'

## 1. 공통 함수

`read_csv_smart`는 인코딩 차이를 흡수하기 위한 로더입니다. 엔트로피 가중치는 자치구 간 차이가 큰 지표에 더 큰 가중치를 부여하는 방식입니다.

In [ ]:
def read_csv_smart(path, **kwargs):
    encodings = ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as exc:
            last_error = exc
    raise last_error

def pick_col(df, candidates=None, fallback_index=None, contains=None, label='column'):
    candidates = candidates or []
    cols = list(df.columns)
    for cand in candidates:
        if cand in cols:
            return cand
    if contains:
        for col in cols:
            text = str(col)
            if all(token in text for token in contains):
                return col
    if fallback_index is not None and fallback_index < len(cols):
        return cols[fallback_index]
    raise KeyError(f'{label} 컬럼을 찾을 수 없습니다. columns={cols}')

def rank_100(series):
    return series.rank(method='average', pct=True) * 100

def minmax_0_1(series):
    s = pd.to_numeric(series, errors='coerce')
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or np.isclose(hi, lo):
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)

def entropy_weights(matrix):
    x = matrix.astype(float).copy()
    x = x.apply(minmax_0_1, axis=0)
    eps = 1e-12
    p = x / (x.sum(axis=0) + eps)
    n = len(x)
    k = 1 / np.log(n)
    entropy = -k * (p * np.log(p + eps)).sum(axis=0)
    diversity = 1 - entropy
    if np.isclose(diversity.sum(), 0):
        weights = pd.Series(1 / len(diversity), index=diversity.index)
    else:
        weights = diversity / diversity.sum()
    return weights, entropy, diversity, x

## 2. 대중교통점수 산출

버스와 지하철 승하차 데이터를 연도-자치구 단위로 합산한 뒤, 인구 대비 승차 강도, 하차 강도, 승하차 불균형을 계산합니다. 이후 기존 `교통_대중교통점수.ipynb`와 동일하게 연도별 백분위 점수의 평균을 사용합니다.

In [ ]:
def transit_wide_to_yearly(df):
    work = df.copy()
    if 'gu_name' not in work.columns:
        work = work.rename(columns={work.columns[1]: 'gu_name'})

    value_cols = [c for c in work.columns if re.match(r'^\d{6}_(geton|getoff|etoff)$', str(c))]
    long_df = work[['gu_name'] + value_cols].melt(id_vars='gu_name', var_name='ym_type', value_name='value')
    long_df['year'] = long_df['ym_type'].astype(str).str[:4].astype(int)
    long_df['io'] = long_df['ym_type'].astype(str).str.split('_').str[-1].str.lower().replace({'etoff': 'getoff'})
    long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce').fillna(0)

    yearly = (
        long_df.groupby(['year', 'gu_name', 'io'], as_index=False)['value'].sum()
        .pivot(index=['year', 'gu_name'], columns='io', values='value')
        .reset_index()
        .rename(columns={'geton': 'geton_total', 'getoff': 'getoff_total'})
    )
    for col in ['geton_total', 'getoff_total']:
        if col not in yearly.columns:
            yearly[col] = 0
    return yearly[['year', 'gu_name', 'geton_total', 'getoff_total']]

def population_long_from_garbage(path):
    raw = read_csv_smart(path)
    gu_col = pick_col(raw, candidates=['자치구'], fallback_index=0, label='자치구')
    pop_cols = [c for c in raw.columns if re.search(r'20\d{2}', str(c))]
    if not pop_cols:
        pop_cols = list(raw.columns[1:4])

    rows = []
    for col in pop_cols:
        year_match = re.search(r'20\d{2}', str(col))
        if not year_match:
            continue
        tmp = raw[[gu_col, col]].copy()
        tmp.columns = ['gu_name', 'population']
        tmp['year'] = int(year_match.group())
        tmp['population'] = pd.to_numeric(tmp['population'], errors='coerce')
        rows.append(tmp[['year', 'gu_name', 'population']])
    return pd.concat(rows, ignore_index=True)

bus = transit_wide_to_yearly(read_csv_smart(BUS_PATH))
subway = transit_wide_to_yearly(read_csv_smart(SUBWAY_PATH))
population = population_long_from_garbage(POPULATION_PATH)

transit = pd.concat([bus, subway], ignore_index=True)
transit = transit.groupby(['year', 'gu_name'], as_index=False)[['geton_total', 'getoff_total']].sum()
transit = transit.merge(population, on=['year', 'gu_name'], how='left')

eps = 1e-9
transit['board_intensity'] = transit['geton_total'] / (transit['population'] + eps)
transit['alight_intensity'] = transit['getoff_total'] / (transit['population'] + eps)
transit['imbalance'] = (transit['geton_total'] - transit['getoff_total']).abs() / (transit['geton_total'] + transit['getoff_total'] + eps)

transit['score_board'] = transit.groupby('year')['board_intensity'].transform(rank_100)
transit['score_alight'] = transit.groupby('year')['alight_intensity'].transform(rank_100)
transit['score_imbalance'] = transit.groupby('year')['imbalance'].transform(rank_100)
transit['transport_stress_score'] = transit[['score_board', 'score_alight', 'score_imbalance']].mean(axis=1)

transport_gu = (
    transit.groupby('gu_name', as_index=False)['transport_stress_score']
    .mean()
    .rename(columns={'transport_stress_score': 'transport_score'})
)
transport_gu.head()

## 3. 출퇴근스트레스점수 집계

월별 출퇴근 스트레스 점수를 자치구 평균으로 집계합니다.

In [ ]:
monthly = read_csv_smart(MONTHLY_PATH)
monthly_gu_col = pick_col(monthly, candidates=['자치구명'], fallback_index=3, label='자치구명')
commute_score_col = pick_col(monthly, candidates=['출퇴근스트레스점수'], fallback_index=10, label='출퇴근스트레스점수')

commute = monthly[[monthly_gu_col, commute_score_col]].copy()
commute.columns = ['gu_name', 'commute_stress_score']
commute['commute_stress_score'] = pd.to_numeric(commute['commute_stress_score'], errors='coerce')

commute_gu = commute.groupby('gu_name', as_index=False)['commute_stress_score'].mean()
commute_gu.head()

## 4. 주차점수와 민원점수 불러오기

`parking_population_complaint_analysis.ipynb`에서 산출한 자치구 평균 요약 파일을 사용합니다.

In [ ]:
parking_complaint = read_csv_smart(PARKING_COMPLAINT_PATH)
parking_complaint = parking_complaint[['gu_name', 'parking_score_100', 'complaint_score_100']].copy()
parking_complaint = parking_complaint.rename(columns={
    'parking_score_100': 'parking_score',
    'complaint_score_100': 'complaint_score',
})
parking_complaint.head()

## 5. 네 지표 결합 및 엔트로피 가중치 계산

네 지표를 각각 0~1로 정규화한 뒤 엔트로피 가중치를 계산합니다. 최종 점수는 25점 만점이며, 높을수록 교통 문제가 큰 자치구로 해석합니다.

In [ ]:
indicator_cols = ['transport_score', 'commute_stress_score', 'parking_score', 'complaint_score']

score_base = (
    transport_gu
    .merge(commute_gu, on='gu_name', how='inner')
    .merge(parking_complaint, on='gu_name', how='inner')
)

missing_count = 25 - score_base['gu_name'].nunique()
print(f'결합 자치구 수: {score_base["gu_name"].nunique()}개')
if missing_count:
    print(f'주의: 기준 25개 대비 {missing_count}개 자치구가 결합되지 않았습니다. 자치구명 인코딩/표기를 확인하세요.')

weights, entropy, diversity, normalized = entropy_weights(score_base[indicator_cols])

weight_summary = pd.DataFrame({
    'indicator': weights.index,
    'entropy': entropy.values,
    'diversity': diversity.values,
    'weight': weights.values,
    'weight_percent': weights.values * 100,
})

score_result = score_base.copy()
for col in indicator_cols:
    score_result[f'{col}_norm'] = normalized[col] * 100

score_result['traffic_entropy_raw_0_1'] = normalized.mul(weights, axis=1).sum(axis=1)
score_result['traffic_score_25'] = score_result['traffic_entropy_raw_0_1'] * 25
score_result['traffic_score_25'] = score_result['traffic_score_25'].round(2)
score_result['rank'] = score_result['traffic_score_25'].rank(method='min', ascending=False).astype(int)

score_result = score_result.sort_values(['rank', 'gu_name']).reset_index(drop=True)
display(weight_summary.round(4))
display(score_result[['rank', 'gu_name', 'traffic_score_25'] + indicator_cols].head(25))

## 6. 결과 저장

In [ ]:
weight_summary.to_csv(WEIGHT_PATH, index=False, encoding='utf-8-sig')
score_result.to_csv(FINAL_SCORE_PATH, index=False, encoding='utf-8-sig')

print(f'가중치 저장: {WEIGHT_PATH}')
print(f'최종 점수 저장: {FINAL_SCORE_PATH}')